### Capturing historical NBA draft selections from basketball reference and building a pipeline that gives us a concatenated table from 2001-2026 of each player drafted from each round.

In [37]:
import pandas as pd
import requests
import certifi
from io import StringIO


url = "https://www.basketball-reference.com/draft/NBA_2026.html"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    verify=certifi.where(),
    timeout=20
)

response.raise_for_status()

tables = pd.read_html(StringIO(response.text))

len(tables)

draft26 = tables[0]

draft26.columns = draft26.columns.get_level_values(-1)

draft26.head(30)

draft_26_cleaner = draft26[["Rk", "Pk", "Tm", "Player", "College"]]

draft_26_cleaner.loc[pd.to_numeric(draft_26_cleaner["Rk"], errors="coerce").isna()]

draft_26_cleaner = draft_26_cleaner.copy()

draft_26_cleaner["round"] = 1

draft_26_cleaner.loc[
    draft_26_cleaner.index > 30, "round"
] = 2

draft_26_cleaner.tail(33)

# pd.to_numeric() attempts to interpret each value in the Rk column
# as a number, including numeric strings like "1" or "30".
#
# errors="coerce" converts values that aren't numeric (such as
# "Rk" or "Round 2") into NaN instead of raising an error.
#
# .notna() creates a Boolean mask:
# True = valid numeric value; False = missing or nonnumeric value.

valid_rows = pd.to_numeric(
    draft_26_cleaner["Rk"],
    errors="coerce"
).notna()

# .loc[valid_rows] retains only rows where the mask is True.
# This removes the separator and repeated header rows.
#
# .copy() creates an independent DataFrame from the filtered result.

draft_26_cleaner = draft_26_cleaner.loc[valid_rows].copy()

draft_26_cleaner.head(35)

,Rk,Pk,Tm,Player,College,round
0,1,1,WAS,AJ Dybantsa,BYU,1
1,2,2,UTA,Darryn Peterson,Kansas,1
2,3,3,MEM,Cameron Boozer,Duke,1
3,4,4,CHI,Caleb Wilson,UNC,1
4,5,5,LAC,Keaton Wagler,Illinois,1
5,6,6,BRK,Mikel Brown Jr.,Louisville,1
6,7,7,SAC,Darius Acuff Jr.,Arkansas,1
7,8,8,ATL,Kingston Flemings,Houston,1
8,9,9,DAL,Morez Johnson Jr.,Michigan,1
9,10,10,MIL,Brayden Burries,Arizona,1


In [ ]:
# Drop Rk because Pk represents the actual draft position.
# We retain Pk and round to identify where each player was selected.

draft_26_cleaner = draft_26_cleaner.drop(columns=["Rk"])

# Assign the draft year to every player in this DataFrame.
# A single value is broadcast across all rows.

draft_26_cleaner["year"] = 2026

# Rename columns to make their meaning clearer.
# The dictionary maps existing column names to their replacements.

draft_26_cleaner = draft_26_cleaner.rename(columns={
    "Pk": "overall_pick",
    "Tm": "team",
    "Player": "player",
    "College": "college"
})

# Reset the index to consecutive row numbers.
# drop=True prevents the old index from becoming another column.

draft_26_cleaner = draft_26_cleaner.reset_index(drop=True)


KeyError: "['Rk'] not found in axis"

In [40]:
draft_26_cleaner.head(14)

,overall_pick,team,player,college,round,year
0,1,WAS,AJ Dybantsa,BYU,1,2026
1,2,UTA,Darryn Peterson,Kansas,1,2026
2,3,MEM,Cameron Boozer,Duke,1,2026
3,4,CHI,Caleb Wilson,UNC,1,2026
4,5,LAC,Keaton Wagler,Illinois,1,2026
5,6,BRK,Mikel Brown Jr.,Louisville,1,2026
6,7,SAC,Darius Acuff Jr.,Arkansas,1,2026
7,8,ATL,Kingston Flemings,Houston,1,2026
8,9,DAL,Morez Johnson Jr.,Michigan,1,2026
9,10,MIL,Brayden Burries,Arizona,1,2026


In [42]:
draft_26_cleaner.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   overall_pick  60 non-null     str  
 1   team          60 non-null     str  
 2   player        60 non-null     str  
 3   college       54 non-null     str  
 4   round         60 non-null     int64
 5   year          60 non-null     int64
dtypes: int64(2), str(4)
memory usage: 2.9 KB


In [48]:
draft_26_cleaner["overall_pick"] = draft_26_cleaner["overall_pick"].astype(int)

draft_26_cleaner.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   overall_pick  60 non-null     int64
 1   team          60 non-null     str  
 2   player        60 non-null     str  
 3   college       54 non-null     str  
 4   round         60 non-null     int64
 5   year          60 non-null     int64
dtypes: int64(3), str(3)
memory usage: 2.9 KB


In [49]:
import pandas as pd
import requests
import certifi
from io import StringIO


url = "https://www.basketball-reference.com/draft/NBA_2001.html"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    verify=certifi.where(),
    timeout=20
)

response.raise_for_status()

tables = pd.read_html(StringIO(response.text))

len(tables)



1

In [51]:
draft01 = tables[0]

draft01.columns = draft01.columns.get_level_values(-1)

draft01.head(30)

draft_01_cleaner = draft01[["Rk", "Pk", "Tm", "Player", "College"]]

draft_01_cleaner.head(33)

,Rk,Pk,Tm,Player,College
0,1,1,WAS,Kwame Brown,NaN
1,2,2,LAC,Tyson Chandler,NaN
2,3,3,ATL,Pau Gasol,NaN
3,4,4,CHI,Eddy Curry,NaN
4,5,5,GSW,Jason Richardson,Michigan State
5,6,6,MEM,Shane Battier,Duke
6,7,7,NJN,Eddie Griffin,Seton Hall
7,8,8,CLE,DeSagana Diop,NaN
8,9,9,DET,Rodney White,UNC Charlotte
9,10,10,BOS,Joe Johnson,Arkansas


In [52]:
draft_01_cleaner.loc[pd.to_numeric(draft_01_cleaner["Rk"], errors="coerce").isna()]

draft_01_cleaner = draft_01_cleaner.copy()

draft_01_cleaner.head(33)

,Rk,Pk,Tm,Player,College
0,1,1,WAS,Kwame Brown,NaN
1,2,2,LAC,Tyson Chandler,NaN
2,3,3,ATL,Pau Gasol,NaN
3,4,4,CHI,Eddy Curry,NaN
4,5,5,GSW,Jason Richardson,Michigan State
5,6,6,MEM,Shane Battier,Duke
6,7,7,NJN,Eddie Griffin,Seton Hall
7,8,8,CLE,DeSagana Diop,NaN
8,9,9,DET,Rodney White,UNC Charlotte
9,10,10,BOS,Joe Johnson,Arkansas


In [54]:
# Find the row where the Player column contains "Round 2".
# This returns a Boolean mask identifying the separator row.
mask = draft_01_cleaner["Player"] == "Round 2"

# Extract the index label of the matching separator row.
# [0] selects the first matching index.
separator_index = draft_01_cleaner.index[mask][0]

# Assign Round 1 as the default.
draft_01_cleaner["round"] = 1

# Rows appearing after the separator belong to Round 2.
# This avoids hardcoding 29 or 30 for different draft years.
draft_01_cleaner.loc[
    draft_01_cleaner.index > separator_index,
    "round"
] = 2

IndexError: index 0 is out of bounds for axis 0 with size 0

In [55]:
# Check whether the Round 2 separator still exists.
# sum() counts the number of True values in the Boolean mask.
print("Round 2 separators:",
      (draft_01_cleaner["Player"] == "Round 2").sum())

# Find rows where Rk is not numeric.
# These may include separator rows or repeated column headers.
invalid_rows = pd.to_numeric(
    draft_01_cleaner["Rk"],
    errors="coerce"
).isna()

# Display those rows without modifying the DataFrame.
display(draft_01_cleaner.loc[invalid_rows])

# Check whether round assignments already exist.
print("\nRound counts:")
print(draft_01_cleaner.groupby("round").size())

Round 2 separators: 0


,Rk,Pk,Tm,Player,College,round



Round counts:
round
1    28
2    29
dtype: int64


In [56]:
# Remove the redundant ranking column.
draft_01_cleaner = draft_01_cleaner.drop(columns=["Rk"])

# Rename columns to match the 2026 draft schema.
draft_01_cleaner = draft_01_cleaner.rename(columns={
    "Pk": "overall_pick",
    "Tm": "team",
    "Player": "player",
    "College": "college"
})

# Convert draft positions and rounds to integers.
draft_01_cleaner["overall_pick"] = draft_01_cleaner["overall_pick"].astype(int)
draft_01_cleaner["round"] = draft_01_cleaner["round"].astype(int)

# Add the draft year to every row.
draft_01_cleaner["year"] = 2001

# Reset the index after removing the unwanted rows.
draft_01_cleaner = draft_01_cleaner.reset_index(drop=True)

# Inspect the result.
draft_01_cleaner.info()

<class 'pandas.DataFrame'>
RangeIndex: 57 entries, 0 to 56
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   overall_pick  57 non-null     int64
 1   team          57 non-null     str  
 2   player        57 non-null     str  
 3   college       45 non-null     str  
 4   round         57 non-null     int64
 5   year          57 non-null     int64
dtypes: int64(3), str(3)
memory usage: 2.8 KB


In [57]:
draft_01_cleaner.head()

,overall_pick,team,player,college,round,year
0,1,WAS,Kwame Brown,NaN,1,2001
1,2,LAC,Tyson Chandler,NaN,1,2001
2,3,ATL,Pau Gasol,NaN,1,2001
3,4,CHI,Eddy Curry,NaN,1,2001
4,5,GSW,Jason Richardson,Michigan State,1,2001


In [58]:
# Confirm the number of rows and data types.
print(draft_01_cleaner.shape)
print(draft_01_cleaner.dtypes)

# Check whether any overall pick appears more than once.
print("Duplicate picks:",
      draft_01_cleaner["overall_pick"].duplicated().sum())

# Confirm the final first-round pick and first second-round pick.
# This checks the round boundary without assuming 30 first-round picks.
print(draft_01_cleaner.groupby("round")["overall_pick"].agg(["min", "max"]))

(57, 6)
overall_pick    int64
team              str
player            str
college           str
round           int64
year            int64
dtype: object
Duplicate picks: 0
       min  max
round          
1        1   28
2       30   58


In [67]:
import pandas as pd
import requests
import certifi
from io import StringIO


def clean_draft(year):

    # 1. BUILD THE URL
    # The f-string inserts the year into the webpage address.
    # For example, year=2001 produces NBA_2001.html.

    url = f"https://www.basketball-reference.com/draft/NBA_{year}.html"

    # 2. EXTRACT THE DATA
    # requests.get() downloads the HTML webpage.
    # raise_for_status() checks whether the request succeeded.

    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        verify=certifi.where(),
        timeout=20
    )

    response.raise_for_status()
    response.encoding = "utf-8"

    # pd.read_html() returns a LIST of DataFrames.
    # [0] selects the first table in that list.

    tables = pd.read_html(StringIO(response.text))
    df = tables[0].copy()

    # 3. CLEAN COLUMN HEADERS
    # Some Basketball-Reference tables have multiple header levels.
    # get_level_values(-1) keeps the lowest level.

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(-1)

    # Select only the five source columns we need.
    # Double brackets return a DataFrame rather than a Series.

    df = df[["Rk", "Pk", "Tm", "Player", "College"]].copy()

    # Reset the index so row positions match index labels.
    df = df.reset_index(drop=True)

    # 4. IDENTIFY THE ROUND 2 SEPARATOR
    # == creates a Boolean mask: True where Player is "Round 2".
    # df.index[mask][0] retrieves the separator's index.

    mask = df["Player"] == "Round 2"
    separator_index = df.index[mask][0]

    # 5. ASSIGN DRAFT ROUNDS
    # First assign Round 1 to every row.
    # Then assign Round 2 to rows after the separator.
    # This works even when the first round has fewer than 30 picks.

    df["round"] = 1
    df.loc[df.index > separator_index, "round"] = 2

    # 6. REMOVE UNWANTED ROWS
    # pd.to_numeric() recognizes numbers stored as strings.
    # errors="coerce" converts nonnumeric values to NaN.
    # .notna() creates a Boolean mask identifying valid numeric rows.
    # .loc[] retains only those rows.

    valid_rows = pd.to_numeric(
        df["Rk"],
        errors="coerce"
    ).notna()

    df = df.loc[valid_rows].copy()

    # 7. DROP THE REDUNDANT COLUMN
    # Rk is unnecessary because Pk identifies the draft position.

    df = df.drop(columns=["Rk"])

    # 8. RENAME COLUMNS
    # The dictionary maps original names to standardized names.

    df = df.rename(columns={
        "Pk": "overall_pick",
        "Tm": "team",
        "Player": "player",
        "College": "college"
    })

    # 9. CONVERT DATA TYPES
    # astype(int) converts numeric strings into integers.

    df["overall_pick"] = df["overall_pick"].astype(int)
    df["round"] = df["round"].astype(int)

    # 10. ADD DRAFT YEAR
    # Assign the function's year argument to every row.

    df["year"] = year

    # 11. RESET INDEX
    # Remove gaps left behind by filtering out unwanted rows.

    df = df.reset_index(drop=True)

    # 12. RETURN THE FINISHED DATAFRAME
    # This makes the cleaned data available outside the function.

    return df


# TEST THE FUNCTION ON TWO DIFFERENT YEARS

draft_2001 = clean_draft(2001)
draft_2026 = clean_draft(2026)

print("2001:", draft_2001.shape)
print("2026:", draft_2026.shape)

print("\n2001 round counts:")
print(draft_2001.groupby("round").size())

print("\n2026 round counts:")
print(draft_2026.groupby("round").size())

display(draft_2001.head())
display(draft_2026.head())

2001: (57, 6)
2026: (60, 6)

2001 round counts:
round
1    28
2    29
dtype: int64

2026 round counts:
round
1    30
2    30
dtype: int64


,overall_pick,team,player,college,round,year
0,1,WAS,Kwame Brown,NaN,1,2001
1,2,LAC,Tyson Chandler,NaN,1,2001
2,3,ATL,Pau Gasol,NaN,1,2001
3,4,CHI,Eddy Curry,NaN,1,2001
4,5,GSW,Jason Richardson,Michigan State,1,2001


,overall_pick,team,player,college,round,year
0,1,WAS,AJ Dybantsa,BYU,1,2026
1,2,UTA,Darryn Peterson,Kansas,1,2026
2,3,MEM,Cameron Boozer,Duke,1,2026
3,4,CHI,Caleb Wilson,UNC,1,2026
4,5,LAC,Keaton Wagler,Illinois,1,2026


In [68]:
import time

# STEP 1: DEFINE THE YEARS
# range() includes 2001 but excludes 2027.
# This gives us 26 draft classes, from 2001 through 2026.

years = range(2001, 2027)

# STEP 2: INITIALIZE AN EMPTY LIST
# This will store one cleaned DataFrame for each year.

draft_dfs = []

# Track any years that fail so we can retry them separately.
failed_years = []

# STEP 3: LOOP THROUGH EACH YEAR
# On each iteration, year takes the next value from range().

for year in years:

    try:
        # Call the function we already created.
        # It downloads, cleans, and returns one year's DataFrame.
        df = clean_draft(year)

        # Add the completed DataFrame to our list.
        # append() adds one object to the end of a list.
        draft_dfs.append(df)

        # Display progress.
        print(f"{year}: {len(df)} picks collected")

    except Exception as e:
        # Record failures without stopping the entire loop.
        # This also captures parsing or cleaning errors.
        failed_years.append(year)
        print(f"{year}: FAILED — {e}")

    # Wait between requests to avoid repeatedly hitting the website.
    time.sleep(5)

# STEP 4: CHECK RESULTS
print("\nCompleted years:", len(draft_dfs))
print("Failed years:", failed_years)

# STEP 5: CONCATENATE
# pd.concat() stacks the yearly DataFrames vertically.
# ignore_index=True creates one continuous index.

if draft_dfs:
    draft_all = pd.concat(draft_dfs, ignore_index=True)

    print("\nCombined shape:", draft_all.shape)
    display(draft_all.head())
else:
    print("No DataFrames collected. Nothing to concatenate.")

2001: 57 picks collected
2002: 57 picks collected
2003: 58 picks collected
2004: 59 picks collected
2005: 60 picks collected
2006: 60 picks collected
2007: 60 picks collected
2008: 60 picks collected
2009: 60 picks collected
2010: 60 picks collected
2011: 60 picks collected
2012: 60 picks collected
2013: 60 picks collected
2014: 60 picks collected
2015: 60 picks collected
2016: 60 picks collected
2017: 60 picks collected
2018: 60 picks collected
2019: 60 picks collected
2020: 60 picks collected
2021: 60 picks collected
2022: 58 picks collected
2023: 58 picks collected
2024: 60 picks collected
2025: 60 picks collected
2026: 60 picks collected

Completed years: 26
Failed years: []

Combined shape: (1547, 6)


,overall_pick,team,player,college,round,year
0,1,WAS,Kwame Brown,NaN,1,2001
1,2,LAC,Tyson Chandler,NaN,1,2001
2,3,ATL,Pau Gasol,NaN,1,2001
3,4,CHI,Eddy Curry,NaN,1,2001
4,5,GSW,Jason Richardson,Michigan State,1,2001


In [69]:
# 1. Confirm all 26 draft years are present.
print("Unique years:", draft_all["year"].nunique())
print("Year range:", draft_all["year"].min(), "-", draft_all["year"].max())

# 2. Confirm the six expected columns.
expected_columns = [
    "overall_pick", "team", "player",
    "college", "round", "year"
]
print("Correct columns:", draft_all.columns.tolist() == expected_columns)

# 3. Check for missing values in essential fields.
# College is intentionally excluded because some players did not attend college.
print("\nMissing essential values:")
print(draft_all[["overall_pick", "team", "player", "round", "year"]].isna().sum())

# 4. Check for duplicate draft selections.
# A pick should appear only once within a given draft year.
print("\nDuplicate year/pick combinations:",
      draft_all.duplicated(subset=["year", "overall_pick"]).sum())

# 5. Confirm rounds contain only 1 and 2.
print("\nRound values:", sorted(draft_all["round"].unique()))

# 6. Inspect the number of selections in each year.
print("\nPicks by year and round:")
print(draft_all.groupby(["year", "round"]).size().unstack(fill_value=0))

# 7. Final shape and data types.
print("\nCombined shape:", draft_all.shape)
print("\nData types:")
print(draft_all.dtypes)

Unique years: 26
Year range: 2001 - 2026
Correct columns: True

Missing essential values:
overall_pick    0
team            0
player          3
round           0
year            0
dtype: int64

Duplicate year/pick combinations: 0

Round values: [np.int64(1), np.int64(2)]

Picks by year and round:
round   1   2
year         
2001   28  29
2002   28  29
2003   29  29
2004   29  30
2005   30  30
2006   30  30
2007   30  30
2008   30  30
2009   30  30
2010   30  30
2011   30  30
2012   30  30
2013   30  30
2014   30  30
2015   30  30
2016   30  30
2017   30  30
2018   30  30
2019   30  30
2020   30  30
2021   30  30
2022   30  28
2023   30  28
2024   30  30
2025   30  30
2026   30  30

Combined shape: (1547, 6)

Data types:
overall_pick    int64
team              str
player            str
college           str
round           int64
year            int64
dtype: object


In [70]:
# Show the three records with missing player names.
# This lets us identify their draft years, teams, and pick numbers.

draft_all.loc[
    draft_all["player"].isna(),
    ["year", "round", "overall_pick", "team", "player"]
]

,year,round,overall_pick,team,player
1425,2024,2,59,PHI,NaN
1426,2024,2,60,PHO,NaN
1486,2025,2,60,NYK,NaN


In [71]:
# Inspect all records around the missing player names.
# Include the college column to see whether other information is missing.

display(
    draft_all.loc[
        ((draft_all["year"] == 2024) & (draft_all["overall_pick"] >= 55))
        |
        ((draft_all["year"] == 2025) & (draft_all["overall_pick"] >= 55))
    ]
)

,overall_pick,team,player,college,round,year
1421,55,LAL,Bronny James,USC,2,2024
1422,56,PHO,Kevin McCullar Jr.,Kansas,2,2024
1423,57,MEM,Ulrich Chomche,NaN,2,2024
1424,58,DAL,Ariel Hukporti,NaN,2,2024
1425,59,PHI,NaN,NaN,2,2024
1426,60,PHO,NaN,NaN,2,2024
1481,55,LAL,Lachlan Olbrich,NaN,2,2025
1482,56,MEM,Will Richard,Florida,2,2025
1483,57,ORL,Max Shulga,VCU,2,2025
1484,58,CLE,Saliou Niang,NaN,2,2025


In [72]:
# Remove records without an actual drafted player.
# .notna() keeps rows where the player field has a value.
# .copy() creates an independent DataFrame.
# reset_index() gives the remaining records continuous row numbers.

draft_all = (
    draft_all.loc[draft_all["player"].notna()]
    .copy()
    .reset_index(drop=True)
)

print("Updated shape:", draft_all.shape)
print("Missing players:", draft_all["player"].isna().sum())

Updated shape: (1544, 6)
Missing players: 0


In [73]:
# FINAL S17 VALIDATION

# 1. Check coverage, shape, and duplicates.
print("Shape:", draft_all.shape)
print("Unique years:", draft_all["year"].nunique())
print("Year range:", draft_all["year"].min(), "-", draft_all["year"].max())

print("Duplicate year/pick combinations:",
      draft_all.duplicated(subset=["year", "overall_pick"]).sum())

# 2. Check missing values in essential columns.
# College is excluded because missing college information is legitimate.
print("\nMissing essential values:")
print(draft_all[["year", "round", "overall_pick", "team", "player"]].isna().sum())

# 3. Check for suspicious character-encoding patterns.
# Ã, Â, and � often indicate incorrectly decoded text.
suspicious = draft_all["player"].str.contains(
    "Ã|Â|�",
    regex=True,
    na=False
)

print("\nPotentially garbled player names:", suspicious.sum())
display(draft_all.loc[suspicious, ["year", "overall_pick", "player"]])

# 4. Display names containing common legitimate accented characters.
# This helps confirm that UTF-8 decoding worked.
accented = draft_all["player"].str.contains(
    r"[À-ÿ]",
    regex=True,
    na=False
)

print("\nNames containing accented characters:", accented.sum())
display(draft_all.loc[accented, ["year", "player"]].head(20))

# 5. Check round distribution by year.
display(draft_all.groupby(["year", "round"]).size().unstack(fill_value=0))

Shape: (1544, 6)
Unique years: 26
Year range: 2001 - 2026
Duplicate year/pick combinations: 0

Missing essential values:
year            0
round           0
overall_pick    0
team            0
player          0
dtype: int64

Potentially garbled player names: 0


,year,overall_pick,player



Names containing accented characters: 47


,year,player
23,2001,Raül López
63,2002,Nenê
72,2002,Jiří Welsch
124,2003,Mickaël Piétrus
179,2004,Rafael Araújo
201,2004,Anderson Varejão
241,2005,Fran Vázquez
253,2005,Francisco García
265,2005,Ricky Sánchez
278,2005,Mickaël Gelabale


round,1,2
year,,
2001,28,29
2002,28,29
2003,29,29
2004,29,30
2005,30,30
2006,30,30
2007,30,30
2008,30,30
2009,30,30


In [75]:
from pathlib import Path

# Path.cwd() is the directory where Jupyter is currently running.
# .parent moves one level up, from Notebooks to your project root.
project_root = Path.cwd().parent

# Point to the existing data/staging directory in your project.
staging_dir = project_root / "data" / "staging" / "NBA Draft"

# Confirm the parent staging directory exists BEFORE writing files.
# This prevents accidentally creating another data folder.
assert (project_root / "data" / "staging").is_dir(), (
    f"Staging directory not found: {project_root / 'data' / 'staging'}"
)

# Create only the NBA Draft folder if necessary.
staging_dir.mkdir(exist_ok=True)

# Save all 26 individual cleaned draft years.
for year, df in draft_all.groupby("year"):

    file_path = staging_dir / f"nba_draft_{year}.csv"

    df.to_csv(file_path, index=False, encoding="utf-8-sig")

# Save the concatenated dataset in the same folder.
combined_path = staging_dir / "All_Drafts.csv"

draft_all.to_csv(
    combined_path,
    index=False,
    encoding="utf-8-sig"
)

# Confirm the actual destination and file count.
print("Saved to:", staging_dir.resolve())
print("CSV files:", len(list(staging_dir.glob("*.csv"))))

Saved to: /Users/jonathantran/Desktop/nba_economics/data/staging/NBA Draft
CSV files: 27
